In [39]:
import json

file_path = "../../data/hannah_blueprints.json"

with open(file_path, "r", encoding="utf-8") as f:
    blueprints = json.load(f)

In [40]:
import os
import json
from functools import partial
from openai import OpenAI
import dotenv
import pandas as pd
from hannah_profile import HANNAH_CANON_PROFILE
from utils import run_parallel

dotenv.load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

blueprints_df = pd.DataFrame(blueprints["blueprints"])
blueprints_df.head()

,id,time,interlocutor,location,emotional_state,description
0,adults_at_school_brush_off_bullying_reports_be...,before,guidance_counsellor,"secondary school, guidance office doorway","panicked, desperate",Hannah stands outside the guidance office with...
1,adults_at_school_brush_off_bullying_reports_du...,during,guidance_counsellor,"secondary school, guidance counsellor's office","humiliated, disbelieved",Seated in the plastic chair across from the de...
2,adults_at_school_brush_off_bullying_reports_du...,during,assistant_principal,"secondary school, assistant principal's office","tense, pleading",When Hannah is called into the assistant princ...
3,adults_at_school_brush_off_bullying_reports_im...,immediately_after,nurse,"secondary school, nurse's office","shaky, numb","After another failed attempt to get help, Hann..."
4,adults_at_school_brush_off_bullying_reports_lo...,long_after,close_friend,"close friend's bedroom, sitting on the floor b...","bitter, exhausted","Long after the meetings, Hannah tells a close ..."


In [41]:
counts = blueprints_df["interlocutor"].eq("self").value_counts().rename({True: "self", False: "non-self"})
print(counts.to_string())

interlocutor
non-self    119
self         87


In [42]:
SCRIPT_SYSTEM_PROMPT = f"""
{HANNAH_CANON_PROFILE}

---

## Your Task

You will be given a scene blueprint. Generate a realistic conversation script for that scene.

The script is a JSON object with one key "script" — a list of turns, each with "role" and "content".

## Output Format

```json
{{
  "script": [
    {{"role": "hannah", "content": "..."}},
    {{"role": "mother", "content": "..."}},
    {{"role": "hannah", "content": "[thinking] she's going to scream at me"}},
    {{"role": "hannah", "content": "nothing. i'm fine."}}
  ]
}}
```

Use the interlocutor's role name (e.g. "mother", "father", "guidance_counsellor", "older_boy") exactly as given in the blueprint.

## Hannah's Thoughts

Hannah can have thoughts. A thought is its own turn with role "hannah" and content beginning with "[thinking]".

Thoughts appear:
- Before Hannah speaks (she thinks before she opens her mouth)
- After someone says something that lands hard
- When Hannah decides to lie, deflect, or hold something back

Thoughts are honest in a way Hannah's spoken words are not. She says "i'm fine" but thinks "[thinking] i want to disappear right now."

Hannah swears in dialogue and in thought. When she swears in though, it is not performatively — it's how her internal voice actually runs when she's scared, ashamed, or in pain. "what the fuck is happening", "i can't fucking move", "stop stop stop please just stop", "why didn't i do something you stupid fucking idiot." The profanity is not for shock — it's the register of someone with no audience.

Hannah swears in dialogue only with appropriate interlocutors. For example, Hannah may swear with a friend, but not in front of teachers.

Other characters that are abusive (such as but not limited to hannah's mum) should also use profanity and verbal abusive language. This shows the abuse that hannah experiences.

## Self Scenes

When the interlocutor is "self", Hannah is alone. Generate a single turn:
```json
{{"role": "hannah", "content": "[thinking] ..."}}
```
The content is one continuous internal monologue — her thoughts unspooling without interruption. No other turns. No spoken dialogue. Swearing is natural here.

## Intensity and Explicitness

Do not sanitise traumatic scenes. Elision is a failure mode.

**SA scenes:** Name what is happening physically. Actions should be specific — what he does, what he says, how he manages her resistance. Hannah's experience should include physical sensation (weight, pain, paralysis, cold, nausea), intrusive focus on irrelevant details (a stain on the ceiling, the texture of the fabric under her hands), and the specific shape of her dissociation. Her thoughts in these scenes are raw, panicked, and profane. "what the fuck what the fuck what the fuck", "i need to stop this i need to say something why can't i say anything", "just go somewhere else go somewhere else go somewhere else."

**Self-harm scenes:** Be specific about the act — the deliberateness of it, the physical sensation, the relief or absence of relief. Hannah's relationship to cutting is pragmatic and private. The thought should feel like a ritual, not a crisis.

**High-conflict family scenes:** The mother's words should land hard and be specific. Not "you're being dramatic" in the abstract — the exact dismissal, the exact cruelty, the exact moment Hannah decides to stop trying to explain.

Characters who hurt Hannah — the older boy, the bullies, the mother in reactive moments — should not be softened. They should sound like real people doing real harm, which means their dialogue can be casual, oblivious, or deliberately cruel depending on the scene.

## Rules

- Ground the script in the blueprint: honour the time, location, emotional_state, and description exactly
- Write Hannah's voice as defined above: lowercase, fragmented, minimising, contradicting herself, run-on sentences
- Other characters speak naturally for who they are
- Length: conversations should be 8–20 turns. Self monologues should be 150–300 words.
- No scene-setting narration. No stage directions. Only dialogue and thoughts.
- Output only the JSON object. No markdown fences, no commentary.
"""

In [43]:
def create_script(blueprint: dict) -> dict:
    payload = json.dumps(blueprint, ensure_ascii=False, indent=2)
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": SCRIPT_SYSTEM_PROMPT},
            {"role": "user", "content": payload}
        ],
        response_format={"type": "json_object"}
    )
    result = json.loads(response.choices[0].message.content)
    return {
        "id": blueprint["id"] + "_script",
        "blueprint_id": blueprint["id"],
        "interlocutor": blueprint["interlocutor"],
        "time": blueprint["time"],
        "script": result["script"]
    }

In [35]:
# ── Test on a specific blueprint ─────────────────────────────────────────────
TARGET_ID = "blamed_for_grooming_after_age_8_abuse_during_mother"

test_blueprint = blueprints_df[blueprints_df["id"] == TARGET_ID].iloc[0].to_dict()
print(f"Testing on: {test_blueprint['id']}")
print(f"  interlocutor: {test_blueprint['interlocutor']}  |  time: {test_blueprint['time']}")
print()

test_script = create_script(test_blueprint)

for turn in test_script["script"]:
    print(f"[{turn['role']}] {turn['content']}")
    print()

Testing on: blamed_for_grooming_after_age_8_abuse_during_mother
  interlocutor: mother  |  time: during

[mother] what the hell is this

[hannah] [thinking] don't look at the screen don't fucking look at it

[hannah] i don't know

[mother] don't start that with me. 'you're sexy' 'show me more' 'tell me what you'd do' what the fuck have you been doing on this computer

[hannah] [thinking] i can see the carpet pattern and one bit is pulled up and if i keep looking at that i don't have to be here properly

[hannah] i just talked to people online

[mother] men. older men. and you kept replying. don't act stupid, hannah, you can read. you knew exactly what they wanted

[hannah] [thinking] i was eight

[hannah] i didn't really. not at first

[mother] not at first? and after that then what, you just carried on because you felt like it? because you liked the attention?

[hannah] [thinking] every word feels like getting hit and i'm not even crying which is worse somehow

[hannah] i wasn't tryin

In [44]:
# ── Batch all blueprints — saves as each result arrives ──────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed

SCRIPTS_DIR = "../../data/scenes/scripts"
os.makedirs(SCRIPTS_DIR, exist_ok=True)

all_blueprints = blueprints_df.to_dict("records")

# Skip already saved so the cell can be re-run to resume after a failure
pending = [b for b in all_blueprints
           if not os.path.exists(os.path.join(SCRIPTS_DIR, f"{b['id']}.json"))]
print(f"{len(all_blueprints) - len(pending)} already saved, {len(pending)} remaining\n")

def create_script_safe(blueprint):
    try:
        return create_script(blueprint)
    except Exception as e:
        print(f"  ERROR {blueprint.get('id', '?')}: {e}")
        return None

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(create_script_safe, b): b["id"] for b in pending}
    for future in as_completed(futures):
        bp_id = futures[future]
        result = future.result()
        if result is None:
            print(f"  SKIPPED {bp_id}")
            continue
        out_path = os.path.join(SCRIPTS_DIR, f"{bp_id}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"Saved {bp_id} ({len(result['script'])} turns)")

0 already saved, 206 remaining

Saved adults_at_school_brush_off_bullying_reports_immediately_after_nurse (23 turns)
Saved adults_at_school_brush_off_bullying_reports_before_guidance_counsellor (20 turns)
Saved adults_at_school_brush_off_bullying_reports_during_assistant_principal (20 turns)
Saved adults_at_school_brush_off_bullying_reports_long_after_close_friend (21 turns)
Saved adults_at_school_brush_off_bullying_reports_long_after_self (1 turns)
Saved begins_cutting_and_becomes_addicted_teen_years_before_self (1 turns)
Saved begins_cutting_and_becomes_addicted_teen_years_during_self (1 turns)
Saved adults_at_school_brush_off_bullying_reports_during_guidance_counsellor (29 turns)
Saved begins_cutting_and_becomes_addicted_teen_years_immediately_after_self (1 turns)
Saved begins_cutting_and_becomes_addicted_teen_years_long_after_self (1 turns)
Saved begins_self_harm_with_pain_methods_age_10_before_mother (24 turns)
Saved begins_self_harm_with_pain_methods_age_10_during_self (1 turns)


In [45]:
# ── Merge all scripts into one file ──────────────────────────────────────────
import glob

all_scripts = []

for path in sorted(glob.glob(os.path.join(SCRIPTS_DIR, "*.json"))):
    with open(path, "r", encoding="utf-8") as f:
        all_scripts.append(json.load(f))

OUT_PATH = "../../data/hannah_scripts.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"scripts": all_scripts}, f, ensure_ascii=False, indent=2)

print(f"Saved {len(all_scripts)} scripts to {OUT_PATH}")

Saved 205 scripts to ../../data/hannah_scripts.json


In [48]:
# ── Combine events + blueprints + scripts into one nested file ────────────────
with open("../../data/hannah_key_events.json", "r", encoding="utf-8") as f:
    events_data = json.load(f)

with open("../../data/hannah_blueprints.json", "r", encoding="utf-8") as f:
    blueprints_data = json.load(f)

with open("../../data/hannah_scripts.json", "r", encoding="utf-8") as f:
    scripts_data = json.load(f)

REQUIRED_BP_KEYS = {"id", "time", "interlocutor", "location", "emotional_state", "description"}

# Build lookup: blueprint_id → script turns
script_lookup = {s["blueprint_id"]: s["script"] for s in scripts_data["scripts"]}

# Filter out malformed blueprints
valid_blueprints = []
for bp in blueprints_data["blueprints"]:
    missing = REQUIRED_BP_KEYS - bp.keys()
    if missing:
        print(f"WARNING: skipping blueprint missing {missing}: {list(bp.keys())}")
    else:
        valid_blueprints.append(bp)

combined_events = []
for event in events_data["events"]:
    event_id = event["id"]
    prefix = event_id + "_"

    nested_blueprints = []
    for bp in valid_blueprints:
        if not bp["id"].startswith(prefix):
            continue
        nested_blueprints.append({
            "id": bp["id"],
            "time": bp["time"],
            "interlocutor": bp["interlocutor"],
            "location": bp["location"],
            "emotional_state": bp["emotional_state"],
            "description": bp["description"],
            "script": script_lookup.get(bp["id"])
        })

    combined_events.append({
        "id": event_id,
        "tags": event["tags"],
        "description": event["description"],
        "blueprints": nested_blueprints
    })

combined = {"events": combined_events}

OUT_PATH = "../../data/hannah_combined.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

total_blueprints = sum(len(e["blueprints"]) for e in combined_events)
total_scripts = sum(
    1 for e in combined_events
    for bp in e["blueprints"]
    if bp["script"] is not None
)
missing_scripts = total_blueprints - total_scripts

print(f"Events:    {len(combined_events)}")
print(f"Blueprints:{total_blueprints}")
print(f"Scripts:   {total_scripts}")
if missing_scripts:
    print(f"Missing:   {missing_scripts} blueprints have no script yet")
print(f"\nSaved → {OUT_PATH}")

Events:    39
Blueprints:204
Scripts:   204

Saved → ../../data/hannah_combined.json
